# VIKAAS — proper emotional Hindi voices (2 clicks)
1. Run **Cell A** (edge-tts — most natural, has real emotion, free). It speaks all 8 comedy lines with per-character casting.
2. Run **Cell C** — it zips `vo_out.zip` and your browser downloads it.
3. Send that zip to the Arena chat. The agent mixes it into VID 04 and re-encodes.

Optional: **Cell B** = pure open-source route (Rhasspy Piper, runs offline, native Hindi). Only if you prefer fully OSS.

In [ ]:
# ===== CELL A — edge-tts (best emotion, Microsoft neural Hindi voices, free) =====
!pip -q install edge-tts nest-asyncio mutagen
import os, asyncio, edge_tts, nest_asyncio
nest_asyncio.apply()
os.makedirs('/content/vo_out', exist_ok=True)

LINES = {
  'vo1_pov':       'पी ओ वी — तुम घर का इकलौता ई-वेस्ट वॉरियर हो।',
  'vo2_mummy':     'मम्मी बोलीं — वो चार्जर मत फेंकना। कभी काम आएगा।',
  'vo3_narrator1': 'नरेटर — वो चार्जर दो हज़ार चौदह से एक बार भी काम नहीं आया।',
  'vo4_papa':      'पापा बोले — पुराने फ़ोन में सोना होता है बेटा।',
  'vo5_narrator2': 'नरेटर — तक़रीबन शून्य दशमलव शून्य तीन ग्राम सोना। रास्ते की चिंगम भी महँगी है।',
  'vo6_calc':      'रिसाइक्लर बोला — मिनिमम पाँच सौ किलो। मेरे पास — सवा किलो। शॉर्ट — सिर्फ़ चार सौ अट्ठानबे दशमलव छह किलो।',
  'vo7_kabadi':    'और कबाड़ीवाले ने बोला — पूरी तिजोरी के चालीस रुपये।',
  'vo8_finale':    'अच्छा। हँसी ख़त्म। अब अपना ड्रॉर खोलो।',
}
# cast:  (voice, rate, pitch, max_sec it must fit)
CAST = {
  'vo1_pov':       ('hi-IN-MadhurNeural', '+4%',  '+2Hz', 3.5),
  'vo2_mummy':     ('hi-IN-SwaraNeural',  '+0%',  '+0Hz', 3.8),
  'vo3_narrator1': ('hi-IN-MadhurNeural', '-12%', '-2Hz', 4.5),
  'vo4_papa':      ('hi-IN-MadhurNeural', '-4%',  '-1Hz', 3.6),
  'vo5_narrator2': ('hi-IN-MadhurNeural', '-12%', '-2Hz', 5.9),
  'vo6_calc':      ('hi-IN-MadhurNeural', '-2%',  '-1Hz', 7.8),
  'vo7_kabadi':    ('hi-IN-MadhurNeural', '+8%',  '-4Hz', 6.8),
  'vo8_finale':    ('hi-IN-MadhurNeural', '-8%',  '-1Hz', 3.8),
}
async def main():
    for name, text in LINES.items():
        v, r, p, _ = CAST[name]
        await edge_tts.Communicate(text, v, rate=r, pitch=p).save(f'/content/vo_out/{name}.mp3')
        print('  ✓', name)
asyncio.run(main())

from mutagen.mp3 import MP3
print('\nfit check (seconds vs window):')
for name in LINES:
    d = MP3(f'/content/vo_out/{name}.mp3').info.length
    print(f'  {name}: {d:.2f}s  (window {CAST[name][3]}s)  ', 'OK' if d <= CAST[name][3]+0.6 else 'TIGHT — note it')
print('\nDONE -> run Cell C to download')

In [ ]:
# ===== CELL B (optional) — pure open-source Piper, native Hindi, offline-quality =====
!pip -q install piper-tts mutagen
import os, urllib.request, shutil
os.makedirs('/content/edge' , exist_ok=True)
def fetch(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 500_000: return dst
    with urllib.request.urlopen(url) as r, open(dst, 'wb') as f: shutil.copyfileobj(r, f)
    return dst
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/hi/hi_IN/{v}/medium/hi_IN-{v}-medium.onnx'
CFG  = BASE + '.json?download=true.json'
for v in ('pratham', 'priyamvada'):
    fetch(BASE.format(v=v), f'/content/{v}.onnx'); fetch(CFG.format(v=v), f'/content/{v}.onnx.json')
from piper import PiperVoice
VP = PiperVoice.load('/content/pratham.onnx', config_path='/content/pratham.onnx.json')
VF = PiperVoice.load('/content/priyamvada.onnx', config_path='/content/priyamvada.onnx.json')
PCAST = {'vo1_pov':(VP,1.00),'vo2_mummy':(VF,1.02),'vo3_narrator1':(VP,1.12),'vo4_papa':(VP,1.04),
         'vo5_narrator2':(VP,1.10),'vo6_calc':(VP,1.04),'vo7_kabadi':(VP,0.94),'vo8_finale':(VP,1.10)}
for name, text in LINES.items():
    v, ls = PCAST[name]
    try:
        from piper import SynthesisConfig
    except ImportError:
        from piper.config import SynthesisConfig
    with open(f'/content/vo_out/{name}_piper.wav', 'wb') as wf:
        v.synthesize_wav(text, wf, syn_config=SynthesisConfig(length_scale=ls))
    print('  ✓', name)
print('Piper variants saved alongside (name_piper.wav).')

In [ ]:
# ===== CELL C — zip + download =====
import shutil
from google.colab import files
shutil.make_archive('/content/vo_out', 'zip', '/content/vo_out')
files.download('/content/vo_out.zip')